# FINE-TUNNING FOR OPENVLA

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 1
# CONFIGURATIONS

import os
os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")  # replace with your repo root
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")

from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import draccus
import torch
import torch.distributed as dist
import tqdm
from accelerate import PartialState
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
from transformers import AutoConfig, AutoImageProcessor
from transformers.modeling_outputs import CausalLMOutputWithPast

import wandb
from prismatic.models.backbones.llm.prompting import PurePromptBuilder, VicunaV15ChatPromptBuilder
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from prismatic.vla.subtrajectory_tokenizer import SubtrajectoryTokenizer
from prismatic.vla.datasets.datasets_custom import RLDSCustomBatchTransform, RLDSDatasetCustom
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics

from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticImageProcessor, PrismaticProcessor

# Sane Defaults
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# # === Utilities ===
# # fmt: off
# def create_vision_transform(vla: nn.Module, input_size: int) -> Callable[[Image.Image], torch.Tensor]:
#     """Gets image transform for the vision encoder."""
#     data_cfg = timm.data.resolve_model_data_config(vla.vision_backbone)
#     data_cfg["input_size"] = (3, input_size, input_size)
#     return timm.data.create_transform(
#         input_size=data_cfg["input_size"],
#         interpolation=data_cfg["interpolation"],
#         mean=data_cfg["mean"],
#         std=data_cfg["std"],
#         crop_pct=1.0,           # Set to 1.0 to disable cropping
#         crop_mode="center",     # Default crop mode --> no-op when `crop_pct == 1.0`
#         is_training=False,      # Disable image_aug when loading transform; handled by RLDS dataloader
#     )
#
# # fmt: on



# fmt: off
vla_path: str = "openvla/openvla-7b"                            # Path to OpenVLA model (on HuggingFace Hub)

# Directory Paths
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")        # Path to Open-X dataset directory
dataset_name: str = "columbia_cairlab_pusht_real"                                # Name of fine-tuning dataset (e.g., `droid_wipe`)
run_root_dir: Path = Path("runs")                               # Path to directory to store logs & checkpoints
adapter_tmp_dir: Path = Path("adapter-tmp")                     # Temporary directory for LoRA weights before fusing

# Fine-tuning Parameters
batch_size: int = 8                                            # Fine-tuning batch size
max_steps: int = 1000 #200_000                                       # Max number of fine-tuning steps
save_steps: int = 500  #5000                                        # Interval for checkpoint saving
learning_rate: float = 2e-5#5e-4                                     # Fine-tuning learning rate
grad_accumulation_steps: int = 1                                # Gradient accumulation steps
image_aug: bool = True                                          # Whether to train with image augmentations
shuffle_buffer_size: int = 10000    #100_000                          # Dataloader shuffle buffer size (can reduce if OOM)
save_latest_checkpoint_only: bool = True                        # Whether to save only one checkpoint per run and
                                                                #   continually overwrite the latest checkpoint
                                                                #   (If False, saves all checkpoints)

# LoRA Arguments
use_lora: bool = True                                           # Whether to use LoRA fine-tuning
lora_rank: int = 32                                             # Rank of LoRA weight matrix
lora_dropout: float = 0.0                                       # Dropout applied to LoRA weights
use_quantization: bool = False                                  # Whether to 4-bit quantize VLA for LoRA fine-tuning
                                                                #   => CAUTION: Reduces memory but hurts performance

# Tracking Parameters
wandb_entity: str = "pollen"          # Name of WandB entity
wandb_project: str = "openvla"        # Name of WandB project                         # Name of entity to log under
run_id_note: Optional[str] = None                               # Extra note for logging, Weights & Biases

# fmt: on


Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft


/home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-30 18:18:45.966420: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-30 18:18:45.966497: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-30 18:18:45.967962: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-30 18:18:45.975681: I tensorflow/core/platform/cpu_feature_guard.cc:182] Thi

Using PUSHT constants:
  NUM_ACTIONS_CHUNK = 1
  ACTION_DIM = 1
  PROPRIO_DIM = 8
  ACTION_PROPRIO_NORMALIZATION_TYPE = bounds_q99
If needed, manually set the correct constants in `prismatic/vla/constants.py`!


In [4]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 2
# PARAMETERS FOR MODEL 


print(f"Fine-tuning OpenVLA Model `{vla_path}` on `{dataset_name}`")

# [Validate] Ensure GPU Available & Set Device / Distributed Context
assert torch.cuda.is_available(), "Fine-tuning assumes at least one GPU is available!"
distributed_state = PartialState()
# torch.cuda.set_device(device_id := distributed_state.local_process_index)
device_id=0 #I will use only one GPU for the momment
device_id = distributed_state.local_process_index
torch.cuda.set_device(device_id)
torch.cuda.empty_cache()

# Configure Unique Experiment ID & Log Directory
exp_id = (
    f"{vla_path.split('/')[-1]}+{dataset_name}"
    f"+b{batch_size * grad_accumulation_steps}"
    f"+lr-{learning_rate}"
    f"+test"
)
if use_lora:
    exp_id += f"+lora-r{lora_rank}+dropout-{lora_dropout}"
if use_quantization:
    exp_id += "+q-4bit"
if run_id_note is not None:
    exp_id += f"--{run_id_note}"
if image_aug:
    exp_id += "--image_aug"

# Start =>> Build Directories
run_dir, adapter_dir = run_root_dir / exp_id, adapter_tmp_dir / exp_id
os.makedirs(run_dir, exist_ok=True)

# Quantization Config =>> only if LoRA fine-tuning
quantization_config = None
if use_quantization:
    assert use_lora, "Quantized training only supported for LoRA fine-tuning!"
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4"
    )

# Register OpenVLA model to HF Auto Classes (not needed if the model is on HF Hub)
AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

# Load OpenVLA Processor and Model using HF AutoClasses
processor = AutoProcessor.from_pretrained(vla_path, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    vla_path,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Device Placement =>> note that BitsAndBytes automatically handles for quantized training
if use_quantization:
    vla = prepare_model_for_kbit_training(vla)
else:
    vla = vla.to(device_id)

    # [LoRA] Wrap Model w/ PEFT `LoraConfig` =>> by default we set `target_modules=all-linear`
    if use_lora:
        lora_config = LoraConfig(
            r=lora_rank,
            lora_alpha=min(lora_rank, 16),
            lora_dropout=lora_dropout,
            target_modules="all-linear",
            init_lora_weights="gaussian",
        )
        vla = get_peft_model(vla, lora_config)
        vla.print_trainable_parameters()

    # Wrap VLA in PyTorch DDP Wrapper for Multi-GPU Training
    # vla = DDP(vla, device_ids=[device_id], find_unused_parameters=True, gradient_as_bucket_view=True)

    # Create Optimizer =>> note that we default to a simple constant learning rate!
    trainable_params = [param for param in vla.parameters() if param.requires_grad]
    optimizer = AdamW(trainable_params, lr=learning_rate)

    # Create Action Tokenizer
    action_tokenizer = SubtrajectoryTokenizer(processor.tokenizer,bins=30,min_action=0,max_action=30)

Fine-tuning OpenVLA Model `openvla/openvla-7b` on `columbia_cairlab_pusht_real`


Loading checkpoint shards: 100%|█████████████████████████████████| 3/3 [00:00<00:00,  4.31it/s]


trainable params: 110,828,288 || all params: 7,652,065,472 || trainable%: 1.4483


In [5]:
# FINE-TUNNING SCRIPT FOR OPENVLA MODEL PART 3
# LOADING DATASET
# Create training and optional validation datasets

# Load Fine-tuning Dataset =>> note that we use an RLDS-formatted dataset following Open X-Embodiment by default.
#   =>> If you want to use a non-RLDS dataset (e.g., a standard PyTorch Dataset) see the following commented block.
#   =>> Note that our training code does not loop over epochs because the RLDS loader does this implicitly; if using
#       your own Dataset, make sure to add the appropriate logic to the training loop!
#
# ---
# from prismatic.vla.datasets import DummyDataset
#
# vla_dataset = DummyDataset(
#     action_tokenizer,
#     processor.tokenizer,
#     image_transform=processor.image_processor.apply_transform,
#     prompt_builder_fn=PurePromptBuilder if "v01" not in vla_path else VicunaV15ChatPromptBuilder,
# )
# ---
batch_transform = RLDSCustomBatchTransform(
    action_tokenizer,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder if "v01" not in vla_path else VicunaV15ChatPromptBuilder,
)
vla_dataset = RLDSDatasetCustom(
    data_root_dir,
    dataset_name,
    batch_transform,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=shuffle_buffer_size,
    image_aug=image_aug,
)

# [Important] Save Dataset Statistics =>> used to de-normalize actions for inference!
if distributed_state.is_main_process:
    save_dataset_statistics(vla_dataset.dataset_statistics, run_dir)

# Create Collator and DataLoader
collator = PaddedCollatorForActionPrediction(
    processor.tokenizer.model_max_length, processor.tokenizer.pad_token_id, padding_side="right"
)
dataloader = DataLoader(
    vla_dataset,
    batch_size=batch_size,
    sampler=None,
    collate_fn=collator,
    num_workers=0,  # Important =>> Set to 0 if using RLDS; TFDS rolls its own parallelism!
)

# Initialize Logging =>> W&B
if distributed_state.is_main_process:
    wandb.init(entity=wandb_entity, project=wandb_project, name=f"ft+{exp_id}")

2026-01-30 18:19:11.268271: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


01/30 [18:19:11] INFO     | >> [*] Loading existing dataset statistics from                       ]8;id=297763;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=715421;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#199\199]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0/dat                  
                          aset_statistics_d6170bf2de88fd222da6c9a2203ee8e1f88e82227a970154e370e5e                  
                          137360b3e.json.                                                                          

2026-01-30 18:19:11.773606: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# columbia_cairlab_pusht_real: =============================================1.000000 #
######################################################################################



01/30 [18:19:12] INFO     | >> [*] Threads per Dataset: [1]                                          ]8;id=959479;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=685479;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#538\538]8;;\

...

                 INFO     | >> [*] Constructing datasets...                                          ]8;id=655000;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=958584;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#542\542]8;;\

2026-01-30 18:19:12.303927: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


01/30 [18:19:13] INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=352105;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=325201;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

01/30 [18:19:14] INFO     | >> [*] Saved dataset statistics file at path                          ]8;id=14525;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=110862;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#284\284]8;;\
                          runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-2e-05+test+lora-r32+d                  
                          ropout-0.0--image_aug/dataset_statistics.json                                            

wandb: Currently logged in as: cataclysme-apocalypse (pollen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
# FINE-TUNNING SCRIPT FOR OPENVLA OFT MODEL PART 4
#TRAINING LOOP  # ==================================================

# Deque to store recent train metrics (used for computing smoothened metrics for gradient accumulation)
recent_losses = deque(maxlen=grad_accumulation_steps)
recent_action_accuracies = deque(maxlen=grad_accumulation_steps)
recent_l1_losses = deque(maxlen=grad_accumulation_steps)

# Train!
with tqdm.tqdm(total=max_steps, leave=False) as progress:
    vla.train()
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(dataloader):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            output: CausalLMOutputWithPast = vla(
                input_ids=batch["input_ids"].to(device_id),
                attention_mask=batch["attention_mask"].to(device_id),
                pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
                labels=batch["labels"],
            )
            loss = output.loss

        # Normalize loss to account for gradient accumulation
        normalized_loss = loss / grad_accumulation_steps

        # Backward pass
        normalized_loss.backward()

        # Compute Accuracy and L1 Loss for Logging
        action_logits = output.logits[:, vla.vision_backbone.featurizer.patch_embed.num_patches : -1]
   
        action_preds = action_logits.argmax(dim=2)
        action_gt = batch["labels"][:, 1:].to(action_preds.device)
 
        # print(f"--- DEBUG: ACCTION GT {action_gt} ---")
        mask = action_gt > action_tokenizer.action_token_begin_idx

        print(f"--- DEBUG: pred shape {action_preds.shape} ---")
        print(f"--- DEBUG: gt shape {action_gt.shape} ---")

        # print(f"--- DEBUG: mask  {mask} ---")
        # print(f"--- DEBUG: TOKEN DEBUT {action_tokenizer.action_token_begin_idx} ---")

        # print(f"--- DEBUG: GT TOKEN {action_gt} ---")

        # Compute Accuracy
        correct_preds = (action_preds == action_gt) & mask
        action_accuracy = correct_preds.sum().float() / mask.sum().float()
        print(f"--- DEBUG: ACCURACY {action_accuracy} ---")


        # Compute L1 Loss on Predicted (Continuous) Actions
        subtraject_ID_pred = torch.tensor(
            action_tokenizer.decode_token_ids_to_actions(action_preds[mask].cpu().numpy())
        )
        subtraject_ID_pred_gt = torch.tensor(
            action_tokenizer.decode_token_ids_to_actions(action_gt[mask].cpu().numpy())
        )

        logits = action_logits.transpose(1, 2)
        logits_masked=action_logits[mask]
        action_gt_masked=action_gt[mask]
        # print(f"--- DEBUG: action_logits shape {action_preds.shape} ---")
        print(f"--- DEBUG: action_logits shape {action_preds[mask]} ---") 
        print(f"--- DEBUG: GT shape {action_gt_masked} ---")
        # action_crossEntropy_loss = torch.nn.functional.cross_entropy(logits, action_gt)
        action_crossEntropy_loss = torch.nn.functional.cross_entropy(logits_masked, action_gt_masked)

        # Store recent train metrics
        recent_losses.append(loss.item())
        recent_action_accuracies.append(action_accuracy.item())
        recent_l1_losses.append(action_crossEntropy_loss.item())

        # Compute gradient step index
        gradient_step_idx = batch_idx // grad_accumulation_steps

        # Compute smoothened train metrics
        #   =>> Equal to current step metrics when not using gradient accumulation
        #   =>> Otherwise, equal to the average of metrics observed over micro-batches used for gradient accumulation
        smoothened_loss = sum(recent_losses) / len(recent_losses)
        smoothened_action_accuracy = sum(recent_action_accuracies) / len(recent_action_accuracies)
        smoothened_l1_loss = sum(recent_l1_losses) / len(recent_l1_losses)

        # Push Metrics to W&B (every 10 gradient steps)
        if distributed_state.is_main_process and gradient_step_idx % 10 == 0:
            wandb.log(
                {
                    "train_loss": smoothened_loss,
                    "subtrajectory_ID_accuracy": smoothened_action_accuracy,
                    "cross_entropy_loss": smoothened_l1_loss,
                },
                step=gradient_step_idx,
            )

        # Optimizer Step
        if (batch_idx + 1) % grad_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            progress.update()

        # Save Model Checkpoint =>> by default, only keeps the latest checkpoint, continually overwriting it!
        if gradient_step_idx > 0 and gradient_step_idx % save_steps == 0:
            if distributed_state.is_main_process:
                print(f"Saving Model Checkpoint for Step {gradient_step_idx}")

                # If LoRA, we first save adapter weights, then merge into full model; otherwise, default save!
                save_dir = adapter_dir if use_lora else run_dir

                # Save Processor & Weights
                processor.save_pretrained(run_dir)
                vla.save_pretrained(save_dir)

            # Wait for processor and adapter weights to be saved by main process
            # dist.barrier()

            # Merge LoRA weights into model backbone for faster inference
            #   =>> Note that merging is slow and can be done post-hoc to speed up training
            if use_lora:
                base_vla = AutoModelForVision2Seq.from_pretrained(
                    vla_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
                )
                merged_vla = PeftModel.from_pretrained(base_vla, adapter_dir)
                merged_vla = merged_vla.merge_and_unload()
                if distributed_state.is_main_process:
                    if save_latest_checkpoint_only:
                        # Overwrite latest checkpoint
                        merged_vla.save_pretrained(run_dir)

                        print(f"Saved Model Checkpoint for Step {gradient_step_idx} at: {run_dir}")
                    else:
                        # Prepare to save checkpoint in new directory
                        checkpoint_dir = Path(str(run_dir) + f"--{gradient_step_idx}_chkpt")
                        os.makedirs(checkpoint_dir, exist_ok=True)

                        # Save dataset statistics to new directory
                        save_dataset_statistics(vla_dataset.dataset_statistics, checkpoint_dir)

                        # Save processor and model weights to new directory
                        processor.save_pretrained(checkpoint_dir)
                        merged_vla.save_pretrained(checkpoint_dir)

                        print(f"Saved Model Checkpoint for Step {gradient_step_idx} at: {checkpoint_dir}")

            # Block on Main Process Checkpointing
            # dist.barrier()

        # Stop training when max_steps is reached
        if gradient_step_idx == max_steps:
            print(f"Max step {max_steps} reached! Stopping training...")
            break

  0%|                                                                 | 0/1000 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
W0000 00:00:1769793556.222417 1691217 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 224 } dim { size: 224 } dim { size: 3 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "AuthenticAMD" model: "241" frequency: 2994 num_cores: 8 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 32768 l2_cache_size: 524288 l

--- DEBUG: pred shape torch.Size([8, 62]) ---
--- DEBUG: gt shape torch.Size([8, 62]) ---
--- DEBUG: ACCURACY 0.0 ---
--- DEBUG: action_logits shape tensor([31872, 31872, 31872, 31872, 31872, 31872, 31872, 31872],
       device='cuda:0') ---
--- DEBUG: GT shape tensor([31986, 31986, 31991, 31988, 31994, 31997, 31976, 31976],
       device='cuda:0') ---


  0%|                                                       | 2/1000 [00:11<1:24:15,  5.07s/it]

--- DEBUG: pred shape torch.Size([8, 62]) ---
--- DEBUG: gt shape torch.Size([8, 62]) ---
--- DEBUG: ACCURACY 0.0 ---
--- DEBUG: action_logits shape tensor([31872, 31872, 31872, 31872, 31872, 31872, 31872, 31872],
       device='cuda:0') ---
--- DEBUG: GT shape tensor([31998, 31988, 31974, 31995, 31998, 31984, 31973, 31981],
       device='cuda:0') ---


  0%|▏                                                        | 3/1000 [00:12<51:36,  3.11s/it]

...

 29%|████████████████▏                                      | 294/1000 [04:03<09:32,  1.23it/s]

--- DEBUG: pred shape torch.Size([8, 62]) ---
--- DEBUG: gt shape torch.Size([8, 62]) ---
--- DEBUG: ACCURACY 1.0 ---
--- DEBUG: action_logits shape tensor([31976, 31980, 31976, 31994, 31994, 31976, 31974, 31974],
       device='cuda:0') ---
--- DEBUG: GT shape tensor([31976, 31980, 31976, 31994, 31994, 31976, 31974, 31974],
       device='cuda:0') ---


 30%|████████████████▏                                      | 295/1000 [04:04<09:25,  1.25it/s]

--- DEBUG: pred shape torch.Size([8, 62]) ---
--- DEBUG: gt shape torch.Size([8, 62]) ---
--- DEBUG: ACCURACY 0.875 ---
--- DEBUG: action_logits shape tensor([31995, 31976, 31998, 31988, 31988,     2, 31976, 31983],
       device='cuda:0') ---
--- DEBUG: GT shape tensor([31995, 31976, 31998, 31988, 31988, 31971, 31976, 31983],
       device='cuda:0') ---


KeyboardInterrupt: 

# INFERENCE OPENVLA

In [3]:
import sys
import os

os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")

import torch

from PIL import Image
import numpy as np
from pathlib import Path
from prismatic.vla.datasets.datasets_custom import RLDSCustomBatchTransform, RLDSDatasetCustom

from transformers import AutoModelForVision2Seq, AutoProcessor


from experiments.robot.openvla_utils import _load_dataset_stats
from prismatic.vla.subtrajectory_tokenizer import SubtrajectoryTokenizer
from prismatic.models.backbones.llm.prompting import PurePromptBuilder


pretrained_checkpoint ="/home/ids/ext-5219/tokenizer/openvla-oft/runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-0.0005+lora-r32+dropout-0.0--image_aug/"

# pretrained_checkpoint ="/home/ids/ext-5219/tokenizer/openvla-oft/runs/openvla-7b+columbia_cairlab_pusht_real+b8+lr-2e-05+test+lora-r32+dropout-0.0--image_aug"
# Instantiate config


# Load OpenVLA policy and inputs processor
processor = AutoProcessor.from_pretrained(pretrained_checkpoint, trust_remote_code=True)
# vla = AutoModelForVision2Seq.from_pretrained(
#     pretrained_checkpoint, 
#     attn_implementation="flash_attention_2",  # [Optional] Requires `flash_attn`
#     torch_dtype=torch.bfloat16, 
#     low_cpu_mem_usage=True, 
#     trust_remote_code=True
# ).to("cuda:0")


vla = AutoModelForVision2Seq.from_pretrained(
    pretrained_checkpoint,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda:0")

_load_dataset_stats(vla, pretrained_checkpoint)

print("✓ Modèle chargé avec succès !")

# Load dataset via RLDSDataset (same as training pipeline)
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")
dataset_name: str = "columbia_cairlab_pusht_real"

# Create batch transform
action_tokenizer_inf = SubtrajectoryTokenizer(processor.tokenizer)
batch_transform_inf = RLDSCustomBatchTransform(
    action_tokenizer_inf,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder,
    use_wrist_image=False,
    use_proprio=False,
)

# Create dataset (this properly handles the data structure)
inference_dataset = RLDSDatasetCustom(
    data_root_dir,
    dataset_name,
    batch_transform_inf,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=100,
    image_aug=False,
    train=True,
)

# Get first sample
sample_iterator = iter(inference_dataset)
sample_dict = next(sample_iterator)

print(f"✓ Dataset loaded. Sample keys: {sample_dict.keys()}")

# Extract ground truth subtrajectory_id (check if it exists)

ground_truth_subtrajectory_id = sample_dict["subtraj_ids"]


# Extract and prepare observation for inference
pixel_values = sample_dict["pixel_values"]

# Convert from tensor to numpy if needed
if hasattr(pixel_values, "numpy"):
    pixel_values = pixel_values.numpy()

# The processor may create multi-channel images (e.g., 6 channels for 2 images)
# Extract only the first 3 channels (primary image)
if pixel_values.shape[0] > 3:
    pixel_values = pixel_values[:3]

# Now pixel_values should be (3, H, W) - transpose to (H, W, 3) using numpy
image_np = np.transpose(pixel_values, (1, 2, 0))

# Denormalize from ImageNet normalization to [0, 255]
if image_np.dtype in [np.float32, np.float64]:
    imagenet_mean = np.array([0.485, 0.456, 0.406])
    imagenet_std = np.array([0.229, 0.224, 0.225])
    image_np = (image_np * imagenet_std[np.newaxis, np.newaxis, :]) + imagenet_mean[np.newaxis, np.newaxis, :]
    image_np = np.clip(image_np, 0, 1)
    image_np = (image_np * 255).astype(np.uint8)
else:
    image_np = image_np.astype(np.uint8)



# Grab image input & format prompt
image: Image.Image = Image.fromarray(image_np)
prompt = "In: What subjtrajectories should the robot take to {<INSTRUCTION>}?\nOut:"


print("✓ Observation préparée avec succès !")
print(f"Image shape: {image_np.shape}, dtype: {image_np.dtype}")

# Generate robot action chunk (sequence of future actions)
print("\n→ Starting inference ...")
# Predict Action (7-DoF; un-normalize for BridgeData V2)
inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)
action = vla.predict_subtraj_ID(action_tokenizer_inf, **inputs, unnorm_key="columbia_cairlab_pusht_real", do_sample=False)


gt_cluster =ground_truth_subtrajectory_id


print(f"GT: {gt_cluster}")
print(f"Pred: {action}")
# S'assurer que les deux sont des tableaux numpy "plats" pour la comparaison
gt_flat = int(gt_cluster.item())
pred_flat = int(action.item())



# Comparaison avec une tolérance pour les flottants
is_match = gt_flat== pred_flat

print(f"\n{'='*60}")
print(f"INFERENCE RESULTS:")
print(f"{'='*60}")

match = "✓ CORRECT" if is_match else "✗ MISMATCH"

print(f"Ground Truth: {gt_flat}")
print(f"Predicted:    {pred_flat}")
print(f"Result:       {match}")



Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft


Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.40it/s]


✓ Modèle chargé avec succès !


02/02 [18:31:07] INFO     | >> Load dataset info from                                           ]8;id=559596;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py\dataset_info.py]8;;\:]8;id=699304;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py#599\599]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

                 WARNING  | >> `FeatureConnector.dtype` is deprecated. Please change your code to use ]8;id=520397;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py\feature.py]8;;\:]8;id=769341;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py#67\67]8;;\
                          NumPy with the field `FeatureConnector.np_dtype` or use TensorFlow with the              
                          field `FeatureConnector.tf_dtype`.                                                       

...

                 INFO     | >> Load dataset info from                                           ]8;id=949426;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py\dataset_info.py]8;;\:]8;id=193082;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py#599\599]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=472057;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=345829;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split train, from                                                                        
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

2026-02-02 18:31:07.629983: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


                 INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=456686;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=568631;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

✓ Dataset loaded. Sample keys: dict_keys(['pixel_values', 'input_ids', 'labels', 'dataset_name', 'subtraj_ids'])
✓ Observation préparée avec succès !
Image shape: (224, 224, 3), dtype: uint8

→ Starting inference ...
TESTTT
GT: [17]
Pred: [10]

INFERENCE RESULTS:
Ground Truth: 17
Predicted:    10
Result:       ✗ MISMATCH
